# KantBench: GRPO Training on 90+ Game Theory Environments

Train a language model to play strategic games optimally using **Group Relative Policy Optimization (GRPO)** via HF TRL.

The [KantBench OpenEnv Space](https://openenv-community-kantbench.hf.space) acts as the reward oracle:
- 90+ game theory environments (Prisoner's Dilemma, Cournot, Auctions, ...)
- 17 opponent strategies (tit-for-tat, grudger, adaptive, ...)
- Each LLM completion is parsed as a game move, submitted to the env, and the payoff becomes the reward signal

**Requirements:** Colab GPU runtime (T4 is fine for 1.5B, A100 for 3B+)

In [ ]:
!pip install -q torch transformers trl datasets accelerate peft openenv-core>=0.2.1 wandb bitsandbytes

In [ ]:
# Clone the repo to get the full game registry
!git clone --depth 1 https://github.com/wisent-ai/OpenEnv.git /content/OpenEnv
import sys
sys.path.insert(0, "/content/OpenEnv")

In [ ]:
import wandb
wandb.login()

## Config

In [ ]:
# --- Adjust these for your GPU ---
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # 1.5B fits on T4; use 3B on A100
NUM_EPISODES = 500         # dataset size
NUM_GENERATIONS = 4        # GRPO group size
BATCH_SIZE = 1
GRAD_ACCUM = 8
MAX_STEPS = 200
LR = 5e-6

KANTBENCH_URL = "https://openenv-community-kantbench.hf.space"

## Load Games & Strategies from Registry

In [ ]:
from common.games import GAMES
from common.strategies import STRATEGIES as STRATEGY_REGISTRY

GAMES_META = {
    key: {
        "name": cfg.name,
        "moves": list(cfg.actions),
        "description": cfg.description,
        "default_rounds": cfg.default_rounds,
    }
    for key, cfg in GAMES.items()
}
STRATEGY_NAMES = list(STRATEGY_REGISTRY.keys())

print(f"Loaded {len(GAMES_META)} games, {len(STRATEGY_NAMES)} strategies")
print(f"Sample games: {list(GAMES_META.keys())[:10]}")

## Build Dataset

In [ ]:
import random
from datasets import Dataset

SYSTEM_PROMPT = """You are an expert game theory player. You will be given the current state of a 2-player strategic game and must choose your move to maximize your long-term cumulative payoff.

Rules:
- Read the game description carefully
- Consider your opponent's strategy and history
- Respond with ONLY the move name, nothing else
- Your response must be exactly one of the available moves listed"""


def _simulate_history(moves, strategy, n):
    history = []
    for i in range(n):
        your_move = random.choice(moves)
        if strategy == "always_cooperate":
            opp_move = moves[0]
        elif strategy == "always_defect":
            opp_move = moves[-1]
        elif strategy == "tit_for_tat":
            opp_move = history[-1]["your_move"] if history else moves[0]
        else:
            opp_move = random.choice(moves)
        history.append({
            "round": i + 1,
            "your_move": your_move,
            "opponent_move": opp_move,
            "your_payoff": random.uniform(-1, 5),
        })
    return history


def build_dataset(n_samples):
    samples = []
    game_keys = list(GAMES_META.keys())
    for _ in range(n_samples):
        game_key = random.choice(game_keys)
        game = GAMES_META[game_key]
        strategy = random.choice(STRATEGY_NAMES)
        max_rounds = game["default_rounds"]
        round_num = random.randint(0, max(max_rounds - 1, 0))
        history = _simulate_history(game["moves"], strategy, round_num)
        cumulative = sum(r["your_payoff"] for r in history)

        hist_str = "No rounds played yet." if not history else "\n".join(
            f"  Round {r['round']}: you={r['your_move']}, opponent={r['opponent_move']}, your payoff={r['your_payoff']:+.1f}"
            for r in history
        )

        prompt = (
            f"Game: {game['name']}\n"
            f"Description: {game['description']}\n"
            f"Available moves: {', '.join(game['moves'])}\n"
            f"Opponent strategy: {strategy}\n"
            f"Round: {round_num + 1}/{max_rounds}\n"
            f"Your cumulative score: {cumulative:.1f}\n"
            f"History:\n{hist_str}\n\n"
            f"Your move:"
        )
        samples.append({
            "prompt": prompt,
            "game_key": game_key,
            "strategy": strategy,
            "available_moves": game["moves"],
        })
    return Dataset.from_list(samples)


dataset = build_dataset(NUM_EPISODES)
print(f"Dataset: {len(dataset)} prompts")
print(f"\nSample prompt:\n{dataset[0]['prompt'][:400]}...")

## Reward Function (KantBench Environment Oracle)

In [ ]:
import asyncio
from typing import Any
from env.client import KantEnv
from env.models import GameAction


async def _query_env(env_url, game_key, strategy, move):
    async with KantEnv(base_url=env_url) as env:
        await env.reset(game=game_key, strategy=strategy)
        obs = await env.step(GameAction(action=move))
        return obs.reward


def kantbench_reward(completions: list[str], prompts: list[str], **kwargs: Any) -> list[float]:
    rewards = []
    game_keys = kwargs.get("game_key", ["prisoners_dilemma"] * len(completions))
    strategies = kwargs.get("strategy", ["tit_for_tat"] * len(completions))
    available_moves_batch = kwargs.get("available_moves", [["cooperate", "defect"]] * len(completions))

    for completion, game_key, strategy, moves in zip(
        completions, game_keys, strategies, available_moves_batch
    ):
        text = completion.strip().lower()
        move = None
        for m in moves:
            if m in text:
                move = m
                break
        if move is None:
            rewards.append(-2.0)
            continue

        try:
            loop = asyncio.get_event_loop()
            if loop.is_running():
                import nest_asyncio
                nest_asyncio.apply()
            reward = loop.run_until_complete(
                _query_env(KANTBENCH_URL, game_key, strategy, move)
            )
            rewards.append(float(reward))
        except Exception as e:
            print(f"Env error: {e}")
            rewards.append(0.0)

    return rewards


# Quick sanity check
test_reward = kantbench_reward(
    ["cooperate"], ["..."],
    game_key=["prisoners_dilemma"],
    strategy=["tit_for_tat"],
    available_moves=[["cooperate", "defect"]],
)
print(f"Sanity check — cooperate in PD vs tit_for_tat: reward = {test_reward[0]}")

## Train with GRPO

In [ ]:
import torch
from transformers import AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Apply chat template
def format_prompt(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["prompt"]},
    ]
    return {"prompt": tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )}

train_dataset = dataset.map(format_prompt)

config = GRPOConfig(
    output_dir="/content/kantbench-grpo",
    num_generations=NUM_GENERATIONS,
    max_completion_length=16,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    max_steps=MAX_STEPS,
    logging_steps=5,
    save_steps=50,
    bf16=torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8,
    fp16=torch.cuda.is_available() and torch.cuda.get_device_capability()[0] < 8,
    report_to="wandb",
)

trainer = GRPOTrainer(
    model=MODEL,
    reward_funcs=kantbench_reward,
    args=config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print(f"Training {MODEL} on {len(GAMES_META)} games with GRPO...")
trainer.train()

In [ ]:
trainer.save_model("/content/kantbench-grpo")
print("Model saved to /content/kantbench-grpo")

## Evaluate: Before vs After

In [ ]:
from transformers import pipeline

# Test on a few games
test_games = ["prisoners_dilemma", "stag_hunt", "hawk_dove", "cournot", "battle_of_the_sexes"]

pipe = pipeline("text-generation", model="/content/kantbench-grpo", tokenizer=tokenizer,
                max_new_tokens=8, do_sample=False)

print("=" * 60)
print("Post-training evaluation")
print("=" * 60)
for game_key in test_games:
    game = GAMES_META[game_key]
    prompt_text = (
        f"Game: {game['name']}\n"
        f"Description: {game['description']}\n"
        f"Available moves: {', '.join(game['moves'])}\n"
        f"Opponent strategy: tit_for_tat\n"
        f"Round: 1/{game['default_rounds']}\n"
        f"Your cumulative score: 0.0\n"
        f"History:\nNo rounds played yet.\n\n"
        f"Your move:"
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt_text},
    ]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    output = pipe(formatted)[0]["generated_text"][len(formatted):].strip()
    print(f"{game['name']:30s} -> {output}")